In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Deteksi otomatis folder utama proyek ("Data Mining-SPK")
current_dir = Path.cwd()
BASE_DIR = current_dir.parent if current_dir.name == "Notebooks" else current_dir

DATA_DIR = BASE_DIR / "Data"
RAW_PATH = DATA_DIR / "Raw" / "data_responden.csv"
CLEAN_DIR = DATA_DIR / "Clean"
SKOR_PATH = CLEAN_DIR / "hasil_skor_kriteria.csv"

print("File skor kriteria ketemu? :", SKOR_PATH.exists())
print("File data responden ketemu?:", RAW_PATH.exists())

File skor kriteria ketemu? : True
File data responden ketemu?: True


# Load Data Skor Kriteria & Hitung Bobot SAW ($W$)

Bobot diambil secara data-driven dari rata-rata 4 pertanyaan "Seberapa penting..." pada data_responden.csv

In [2]:
# 1. Baca data skor kriteria (C1 - C4)
analysis = pd.read_csv(SKOR_PATH)

# 2. Baca data mentah untuk mengambil 4 kolom pertanyaan "Seberapa penting..."
df_raw = pd.read_csv(RAW_PATH)
df_raw.columns = [str(c).strip() for c in df_raw.columns]
if "Timestamp" in df_raw.columns:
    df_raw = df_raw[df_raw["Timestamp"].notna()].copy()
df_raw = df_raw.drop_duplicates().copy()

def find_col(df_target, fragment):
    matches = [c for c in df_target.columns if fragment.lower() in c.lower()]
    if not matches:
        raise KeyError(f"Kolom tidak ditemukan untuk kata kunci: {fragment}")
    return matches[0]

col_imp_physical = find_col(df_raw, "penting gaya hidup sehat")
col_imp_mental = find_col(df_raw, "penting kondisi mental")
col_imp_learning = find_col(df_raw, "penting pola belajar")
col_imp_academic = find_col(df_raw, "penting performa akademik")

# Ambil baris yang sesuai dengan index data valid pada 'analysis'
importance = pd.DataFrame({
    "C1_Jasmani": pd.to_numeric(df_raw[col_imp_physical], errors="coerce"),
    "C2_Mental": pd.to_numeric(df_raw[col_imp_mental], errors="coerce"),
    "C3_Pola_Belajar": pd.to_numeric(df_raw[col_imp_learning], errors="coerce"),
    "C4_Akademik": pd.to_numeric(df_raw[col_imp_academic], errors="coerce")
}).loc[analysis.index]

# Hitung rata-rata dan normalisasi bobot (Total = 1)
importance_means = importance.mean()
weights = importance_means / importance_means.sum()

print("=== RATA-RATA TINGKAT KEPENTINGAN ===")
display(importance_means.to_frame("Rata-rata"))

print("\n=== BOBOT KRITERIA SAW (W) ===")
display(weights.to_frame("Bobot"))
print("Total Bobot:", round(float(weights.sum()), 6))

=== RATA-RATA TINGKAT KEPENTINGAN ===


,Rata-rata
C1_Jasmani,4.486486
C2_Mental,4.432432
C3_Pola_Belajar,4.432432
C4_Akademik,4.378378



=== BOBOT KRITERIA SAW (W) ===


,Bobot
C1_Jasmani,0.253049
C2_Mental,0.250000
C3_Pola_Belajar,0.250000
C4_Akademik,0.246951


Total Bobot: 1.0


# Normalisasi Matriks & Perankingan SAW ($V_i$)



In [3]:
saw_input = analysis[weights.index].copy()

# 1. Normalisasi Benefit: r_ij = x_ij / max(x_j)
denominators = saw_input.max(axis=0).replace(0, np.nan)
saw_normalized = saw_input.divide(denominators, axis=1)

# 2. Perhitungan Nilai Preferensi: V_i = sum(w_j * r_ij)
saw_score = saw_normalized.mul(weights, axis=1).sum(axis=1)

# 3. Gabungkan hasil dan buat peringkat
ranking = saw_input.copy()
ranking["Skor_SAW"] = saw_score
ranking["Peringkat"] = ranking["Skor_SAW"].rank(method="min", ascending=False).astype(int)
ranking = ranking.sort_values(["Peringkat", "Skor_SAW"], ascending=[True, False])

print("=== TOP 10 HASIL PERANKINGAN SAW ===")
display(ranking.head(10))

=== TOP 10 HASIL PERANKINGAN SAW ===


,C1_Jasmani,C2_Mental,C3_Pola_Belajar,C4_Akademik,Skor_SAW,Peringkat
28,4.0,4.8,3.0,5.000000,0.927500,1
5,3.2,5.0,3.6,5.000000,0.924390,2
8,4.0,4.0,3.0,4.333333,0.854573,3
0,3.0,3.6,3.6,4.750000,0.829390,4
16,2.8,2.8,4.0,4.750000,0.801738,5
17,3.6,3.0,2.8,4.666667,0.783232,6
6,3.2,3.0,2.8,4.750000,0.762043,7
10,3.0,2.0,3.4,5.000000,0.749238,8
1,3.8,3.2,2.4,4.000000,0.747957,9
3,2.8,3.8,2.6,4.250000,0.739543,10


# Simpan Output Hasil SAW ke Folder Data/Clean

In [4]:
OUTPUT_SAW_PATH = CLEAN_DIR / "hasil_ranking_saw.csv"
ranking.to_csv(OUTPUT_SAW_PATH, index=False)

print("File ranking SAW berhasil disimpan di:")
print("->", OUTPUT_SAW_PATH)

File ranking SAW berhasil disimpan di:
-> c:\Users\daniel\Desktop\Data Mining-SPK\Data\Clean\hasil_ranking_saw.csv
